In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import pandas as pd

df = pd.read_csv("mng_utoronto_2021_clinical_data.tsv", sep="\t")

df.head()

In [ ]:
df.columns

In [ ]:
import pandas as pd
import sqlite3
# read TSV
df = pd.read_csv("mng_utoronto_2021_clinical_data.tsv", sep="\t")
# create sqlite database
conn = sqlite3.connect("genomics.db")
# convert dataframe to SQL table
df.to_sql("oncology_samples",
          conn,
          if_exists="replace",
          index=False)

print("Table created!")



In [ ]:
query = """
SELECT [Molecular Group],
COUNT(*) AS patient_count
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY patient_count DESC
"""

pd.read_sql(query, conn)


In [ ]:
query = """
SELECT [Molecular Group],
AVG([Mutation Count]) AS avg_mutation_count
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY avg_mutation_count DESC
"""

pd.read_sql(query, conn)


In [ ]:
query = """
SELECT [Molecular Group],
AVG([TMB (nonsynonymous)]) AS avg_tmb
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY avg_tmb DESC
"""

pd.read_sql(query, conn)


In [ ]:
query = """
SELECT [Molecular Group],
[Tumor Recurrence],
COUNT(*) AS count
FROM oncology_samples
GROUP BY [Molecular Group],
[Tumor Recurrence]
ORDER BY count DESC
"""

pd.read_sql(query, conn)

In [ ]:
query = """
SELECT
    [Molecular Group],
    ROUND(
        100.0 * SUM(
            CASE
                WHEN [Tumor Recurrence] = 'yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS recurrence_percentage
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY recurrence_percentage DESC
"""

pd.read_sql(query, conn)


In [ ]:
query = """
SELECT [Molecular Group],
[World Health Organization (WHO) Grade],
COUNT(*) AS count
FROM oncology_samples
GROUP BY [Molecular Group],
[World Health Organization (WHO) Grade]
"""

pd.read_sql(query, conn)

In [ ]:
query = """
SELECT
    [Molecular Group],
    [World Health Organization (WHO) Grade],
    COUNT(*) AS sample_count,
    ROUND(
        100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER (
            PARTITION BY [Molecular Group]
        ),
        2
    ) AS percentage
FROM oncology_samples
GROUP BY
    [Molecular Group],
    [World Health Organization (WHO) Grade]
ORDER BY
    [Molecular Group],
    percentage DESC
"""

pd.read_sql(query, conn)

In [ ]:
import matplotlib.pyplot as plt

query = """
SELECT
    [Molecular Group],
    ROUND(
        100.0 * SUM(
            CASE
                WHEN [Tumor Recurrence] = 'yes' THEN 1
                ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS recurrence_percentage
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY recurrence_percentage DESC
"""

df_plot = pd.read_sql(query, conn)

plt.figure(figsize=(8,5))
plt.bar(df_plot["Molecular Group"],
        df_plot["recurrence_percentage"])

plt.xlabel("Molecular Group")
plt.ylabel("Recurrence Percentage")
plt.title("Tumor Recurrence by Molecular Group")
plt.xticks(rotation=45)

plt.show()

In [ ]:
query = """
SELECT
    [Molecular Group],
    AVG([TMB (nonsynonymous)]) AS avg_tmb
FROM oncology_samples
GROUP BY [Molecular Group]
ORDER BY avg_tmb DESC
"""

df_plot = pd.read_sql(query, conn)

plt.figure(figsize=(8,5))
plt.bar(df_plot["Molecular Group"],
        df_plot["avg_tmb"])

plt.xlabel("Molecular Group")
plt.ylabel("Average TMB")
plt.title("Average Tumor Mutational Burden by Molecular Group")
plt.xticks(rotation=45)

plt.show()

In [ ]:
query = """
SELECT
    [Molecular Group],
    [World Health Organization (WHO) Grade],
    COUNT(*) AS count
FROM oncology_samples
GROUP BY
    [Molecular Group],
    [World Health Organization (WHO) Grade]
"""

df_grade = pd.read_sql(query, conn)

pivot_df = df_grade.pivot(
    index="Molecular Group",
    columns="World Health Organization (WHO) Grade",
    values="count"
)

pivot_df.plot(kind="bar", stacked=True)

plt.xlabel("Molecular Group")
plt.ylabel("Sample Count")
plt.title("WHO Grade Distribution Across Molecular Groups")
plt.xticks(rotation=45)

plt.show()